# Advanced Python & OOP -- Practice Exercises & Challenge Projects (Solved)
Student: Newana Tandukar
Day: 5-6

Solutions for every exercise in `advanced-python-oop-questions.ipynb`,
in the same order. Run top to bottom with the `Python 3 (ipykernel)`
kernel; a few cells in Chapters 1 and 2 read `app.log` in this folder.

# Chapter 1: Comprehensions

## Practice Exercises

### Easy

### 1. Cubes of 1-10

In [1]:
cubes = [n ** 3 for n in range(1, 11)]
print(cubes)

[1, 8, 27, 64, 125, 216, 343, 512, 729, 1000]


### 2. Strip surrounding whitespace from strings

In [2]:
raw_strings = ["  hi  ", "bye", "  yo"]
stripped = [s.strip() for s in raw_strings]
print(stripped)

['hi', 'bye', 'yo']


### 3. Unique vowels in a string

In [3]:
text = "education is powerful"
vowels = {ch for ch in text if ch in "aeiou"}
print(sorted(vowels))

['a', 'e', 'i', 'o', 'u']


### Medium

### 1. Map each number to whether it's even

In [4]:
nums = [4, 7, 10, 13, 16, 19]
is_even = {n: n % 2 == 0 for n in nums}
print(is_even)

{4: True, 7: False, 10: True, 13: False, 16: True, 19: False}


### 2. Word length dict, only for words longer than 3 characters

In [5]:
words = ["a", "cat", "python", "programming", "io", "code"]
word_lengths = {w: len(w) for w in words if len(w) > 3}
print(word_lengths)

{'python': 6, 'programming': 11, 'code': 4}


### 3. Sum of squares 1-100 using a generator expression (no list built)

In [6]:
total_of_squares = sum(n ** 2 for n in range(1, 101))
print(total_of_squares)

338350


### Hard

### 1. Flatten a 2D list, keeping only even values

In [7]:
matrix = [[1, 2], [3, 4], [5, 6]]
flat_evens = [x for row in matrix for x in row if x % 2 == 0]
print(flat_evens)

[2, 4, 6]


### 2. Invert a dict with a duplicate value

In [8]:
original = {"a": 1, "b": 2, "c": 2}
inverted = {value: key for key, value in original.items()}
print(inverted)
# b and c both map to 2, so when inverting, the pair processed later in
# dict.items() order ("c": 2) overwrites the earlier one ("b": 2) as the
# value for key 2. Since Python 3.7 dicts preserve insertion order, that
# means the LAST key sharing a value survives the inversion, and the
# others are silently lost. Here 2 ends up mapped to "c", not "b".

{1: 'a', 2: 'c'}


### 3. Count log lines per level

In [9]:
log_lines = [
    "2026-06-25 10:00 INFO user=alice action=login",
    "2026-06-25 10:01 ERROR user=bob action=login msg=timeout",
    "2026-06-25 10:02 WARN user=alice action=upload",
    "2026-06-25 10:03 ERROR user=carol action=login msg=denied",
    "2026-06-25 10:04 INFO user=bob action=logout",
]

levels = ["INFO", "WARN", "ERROR"]
level_counts = {lvl: sum(1 for line in log_lines if lvl in line) for lvl in levels}
print(level_counts)

{'INFO': 2, 'WARN': 1, 'ERROR': 2}


## Challenge Project: Mini Log Analyzer

## Challenge Project: Mini Log Analyzer

In [10]:
log_path = "app.log"

with open(log_path) as f:
    lines = f.read().splitlines()


def field(line, key):
    """Return the value of key=... in a log line, or None if absent."""
    for token in line.split():
        if token.startswith(key + "="):
            return token[len(key) + 1:]
    return None


def analyze_log(lines, level_filter=None):
    # 1. Errors only (lazy generator; list() materialises only when needed)
    errors = list(ln for ln in lines if "ERROR" in ln)

    # 2. Unique users (set comprehension), sorted for stable output
    users = sorted({field(ln, "user") for ln in lines if field(ln, "user")})

    # 3. Level counts (dict comprehension over the known levels)
    levels = ["INFO", "WARN", "ERROR"]
    counts = {lvl: sum(1 for ln in lines if lvl in ln) for lvl in levels}

    # 4. Login failures: (user, msg) pairs for ERROR lines with action=login
    login_failures = [
        (field(ln, "user"), field(ln, "msg"))
        for ln in lines
        if "ERROR" in ln and field(ln, "action") == "login"
    ]

    if level_filter:
        lines = [ln for ln in lines if level_filter in ln]

    return {
        "errors": errors,
        "unique_users": users,
        "level_counts": counts,
        "login_failures": login_failures,
        "filtered": lines if level_filter else None,
    }


report = analyze_log(lines)
print("Errors:", report["errors"])
print("Unique users:", report["unique_users"])
print("Level counts:", report["level_counts"])
print("Login failures:", report["login_failures"])

# Stretch goal: --level filter
only_errors_report = analyze_log(lines, level_filter="ERROR")
print("Filtered to ERROR only:", only_errors_report["filtered"])

Errors: ['2026-06-25 10:01 ERROR user=bob action=login msg=timeout', '2026-06-25 10:03 ERROR user=carol action=login msg=denied', '2026-06-25 10:07 ERROR user=dave action=login msg=timeout']
Unique users: ['alice', 'bob', 'carol', 'dave']
Level counts: {'INFO': 3, 'WARN': 2, 'ERROR': 3}
Login failures: [('bob', 'timeout'), ('carol', 'denied'), ('dave', 'timeout')]
Filtered to ERROR only: ['2026-06-25 10:01 ERROR user=bob action=login msg=timeout', '2026-06-25 10:03 ERROR user=carol action=login msg=denied', '2026-06-25 10:07 ERROR user=dave action=login msg=timeout']


# Chapter 2: Iterators & Generators

## Practice Exercises

### Easy

### 1. Drain a list iterator by hand with next()/try/except (no `for`)

In [11]:
numbers = [5, 10, 15, 20]
it = iter(numbers)

while True:
    try:
        print(next(it))
    except StopIteration:
        break

5
10
15
20


### 2. Generator function count_up(n)

In [12]:
def count_up(n):
    for i in range(1, n + 1):
        yield i


for value in count_up(5):
    print(value)

1
2
3
4
5


### Medium

### 3. even_numbers(limit) generator

In [13]:
def even_numbers(limit):
    n = 2
    while n <= limit:
        yield n
        n += 2


print(list(even_numbers(10)))

[2, 4, 6, 8, 10]


### 4. List comprehension vs generator expression memory

In [14]:
import sys

source_numbers = list(range(100_000))

squares_list = [n * n for n in source_numbers]
squares_gen = (n * n for n in source_numbers)

print("list comprehension size:", sys.getsizeof(squares_list), "bytes")
print("generator expression size:", sys.getsizeof(squares_gen), "bytes")
# The list comprehension materialises every squared value immediately, so
# its size scales with how many items are in source_numbers. The generator
# expression only stores the loop/expression recipe and its current
# position, so its size stays tiny and constant no matter how large
# source_numbers is.

list comprehension size: 800984 bytes
generator expression size: 112 bytes


### 5. Sliding window with previous, current, AND next value

In [15]:
def my_generator():
    yield 10
    yield 20
    yield 30
    yield 40


gen = my_generator()
prev_value = None
current_value = next(gen)
next_value = next(gen, None)

while current_value is not None:
    print(f"Previous: {prev_value}, Current: {current_value}, Next: {next_value}")
    prev_value = current_value
    current_value = next_value
    next_value = next(gen, None)

Previous: None, Current: 10, Next: 20
Previous: 10, Current: 20, Next: 30
Previous: 20, Current: 30, Next: 40
Previous: 30, Current: 40, Next: None


### Hard

### 6. Number class made safe with a limit

In [16]:
class Number:
    def __init__(self, limit):
        self.limit = limit

    def __iter__(self):
        self.n = 2
        return self

    def __next__(self):
        if self.n > self.limit:
            raise StopIteration
        x = self.n
        self.n += 2
        return x


print(list(Number(20)))

[2, 4, 6, 8, 10, 12, 14, 16, 18, 20]


### 7. read_chunks(path, n): yield a file n lines at a time

In [17]:
def read_chunks(path, n):
    with open(path) as f:
        chunk = []
        for line in f:
            chunk.append(line.rstrip("\n"))
            if len(chunk) == n:
                yield chunk
                chunk = []
        if chunk:
            yield chunk


for batch in read_chunks("app.log", 2):
    print(batch)

['2026-06-25 10:00 INFO user=alice action=login', '2026-06-25 10:01 ERROR user=bob action=login msg=timeout']
['2026-06-25 10:02 WARN user=alice action=upload', '2026-06-25 10:03 ERROR user=carol action=login msg=denied']
['2026-06-25 10:04 INFO user=bob action=logout', '2026-06-25 10:05 INFO user=carol action=login']
['2026-06-25 10:06 WARN user=dave action=upload', '2026-06-25 10:07 ERROR user=dave action=login msg=timeout']


## Challenge Project: Memory-Efficient Log/CSV Processor + Infinite Number Generator

### Part A: streaming pipeline

In [18]:
def read_lines(path):
    with open(path) as f:
        for line in f:
            yield line.rstrip("\n")


def parse(lines):
    for line in lines:
        parts = line.split()
        record = {"date": parts[0], "time": parts[1], "level": parts[2]}
        for token in parts[3:]:
            key, _, value = token.partition("=")
            record[key] = value
        yield record


def only_errors(records):
    for record in records:
        if record["level"] == "ERROR":
            yield record


for rec in only_errors(parse(read_lines("app.log"))):
    print(rec)

{'date': '2026-06-25', 'time': '10:01', 'level': 'ERROR', 'user': 'bob', 'action': 'login', 'msg': 'timeout'}
{'date': '2026-06-25', 'time': '10:03', 'level': 'ERROR', 'user': 'carol', 'action': 'login', 'msg': 'denied'}
{'date': '2026-06-25', 'time': '10:07', 'level': 'ERROR', 'user': 'dave', 'action': 'login', 'msg': 'timeout'}


### Part B: infinite generator + take()

In [19]:
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b


def take(gen, n):
    result = []
    it = iter(gen)
    for _ in range(n):
        try:
            result.append(next(it))
        except StopIteration:
            break
    return result


def take_while(gen, predicate):
    for value in gen:
        if not predicate(value):
            break
        yield value


print(take(fibonacci(), 10))
print(list(take_while(fibonacci(), lambda x: x < 20)))

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
[0, 1, 1, 2, 3, 5, 8, 13]


# Chapter 3: Functional Programming

## Practice Exercises

### Easy

### 1. lambda vs def

In [20]:
triple = lambda n: n * 3
print(triple(7))


def triple_def(n):
    return n * 3


print(triple_def(7))
# PEP 8 says not to assign a lambda to a name -- if it needs a name, use
# `def`. `triple_def` keeps a real name in tracebacks/introspection;
# `triple` shows up as `<lambda>` and gains nothing over the `def` form.

21
21


### 2. map/filter vs comprehensions

In [21]:
nums = [5, 12, 17, 22, 30]

squares_map = list(map(lambda x: x ** 2, nums))
divisible_by_3 = list(filter(lambda x: x % 3 == 0, nums))
print("map squares:", squares_map)
print("filter div-by-3:", divisible_by_3)

squares_comp = [x ** 2 for x in nums]
divisible_by_3_comp = [x for x in nums if x % 3 == 0]
print("comprehension squares:", squares_comp)
print("comprehension div-by-3:", divisible_by_3_comp)

map squares: [25, 144, 289, 484, 900]
filter div-by-3: [12, 30]
comprehension squares: [25, 144, 289, 484, 900]
comprehension div-by-3: [12, 30]


### Medium

### 1. Sort words three ways

In [22]:
words = ["banana", "kiwi", "apple", "fig"]

alphabetical = sorted(words)
by_length = sorted(words, key=len)
by_length_desc = sorted(words, key=len, reverse=True)

print("alphabetical:", alphabetical)
print("by length:", by_length)
print("by length desc:", by_length_desc)

alphabetical: ['apple', 'banana', 'fig', 'kiwi']
by length: ['fig', 'kiwi', 'apple', 'banana']
by length desc: ['banana', 'apple', 'kiwi', 'fig']


### 2. Sort people by age, find oldest/youngest

In [23]:
people = [
    {"name": "A", "age": 30},
    {"name": "B", "age": 25},
    {"name": "C", "age": 40},
]

by_age_asc = sorted(people, key=lambda p: p["age"])
oldest = max(people, key=lambda p: p["age"])
youngest = min(people, key=lambda p: p["age"])

print("by age:", by_age_asc)
print("oldest:", oldest)
print("youngest:", youngest)

by age: [{'name': 'B', 'age': 25}, {'name': 'A', 'age': 30}, {'name': 'C', 'age': 40}]
oldest: {'name': 'C', 'age': 40}
youngest: {'name': 'B', 'age': 25}


### 3. Dispatch table calculator

In [24]:
ops = {
    "add": lambda a, b: a + b,
    "mul": lambda a, b: a * b,
    "sub": lambda a, b: a - b,
}


def calc(op, a, b):
    return ops[op](a, b)


print(calc("add", 4, 5))
print(calc("mul", 4, 5))
print(calc("sub", 4, 5))

9
20
-1


### Hard

### 1. reduce for product and flatten

In [25]:
from functools import reduce

product = reduce(lambda a, b: a * b, [1, 2, 3, 4, 5])
print("product:", product)

flattened = reduce(lambda a, b: a + b, [[1, 2], [3, 4], [5]])
print("flattened:", flattened)
# sum() only knows how to add numbers together (it starts from 0 and adds
# each item); it can't multiply, and it can't concatenate lists into a
# flat list either. reduce() is the general-purpose fold that lets you
# supply ANY combining function, which is why it -- not sum() -- can do
# both a running product and a list-flattening concatenation.

product: 120
flattened: [1, 2, 3, 4, 5]


### 2. my_max using reduce

In [26]:
from functools import reduce


def my_max(items, key):
    return reduce(lambda a, b: a if key(a) >= key(b) else b, items)


oldest_person = my_max(people, key=lambda p: p["age"])
print(oldest_person)

{'name': 'C', 'age': 40}


## Challenge Project: Mini Sales Data Pipeline (parse → map → filter → reduce)

### Sales pipeline

In [27]:
from functools import reduce

sales = [
    {"product": "pen", "qty": 10, "price": 1.5},
    {"product": "laptop", "qty": 2, "price": 800.0},
    {"product": "pencil", "qty": 50, "price": 0.5},
    {"product": "monitor", "qty": 3, "price": 150.0},
    {"product": "eraser", "qty": 25, "price": 0.2},
]

# 1. map: add a "total" key to every record
with_totals = list(map(lambda r: {**r, "total": r["qty"] * r["price"]}, sales))

# 2. filter: keep only records with total >= 50
big_sales = list(filter(lambda r: r["total"] >= 50, with_totals))

# 3. reduce: grand total revenue of the filtered records
total_revenue = reduce(lambda acc, r: acc + r["total"], big_sales, 0)

# 4. sorted: order by total, highest first
ranked = sorted(big_sales, key=lambda r: r["total"], reverse=True)

print("--- Sales Report ---")
for r in ranked:
    print(f"{r['product']:8s} total: {r['total']:.2f}")
print(f"Grand total revenue: {total_revenue:.2f}")

--- Sales Report ---
laptop   total: 1600.00
monitor  total: 450.00
Grand total revenue: 2050.00


### Alternative: dict-of-lambdas calculator

In [28]:
calc_ops = {
    "+": lambda a, b: a + b,
    "-": lambda a, b: a - b,
    "*": lambda a, b: a * b,
    "/": lambda a, b: a / b if b != 0 else "error: divide by zero",
}


def calc_v2(a, op, b):
    if op not in calc_ops:
        return f"error: unknown operator '{op}'"
    return calc_ops[op](a, b)


print(calc_v2(10, "+", 5))
print(calc_v2(10, "/", 0))
print(calc_v2(10, "%", 5))

15
error: divide by zero
error: unknown operator '%'


# Chapter 4: Decorators & Properties

## Practice Exercises

### Easy

### 1. @shout decorator

In [29]:
import functools


def shout(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"{result.upper()}!!!")
        return result
    return wrapper


@shout
def say_hi():
    return "hi"


say_hi()

HI!!!


'hi'

### 2. @banner decorator

In [30]:
def banner(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print("=" * 40)
        result = func(*args, **kwargs)
        print("=" * 40)
        return result
    return wrapper


@banner
def greet(name):
    print(f"Hello, {name}!")


greet("Ram")

Hello, Ram!


### Medium

### 1. @timer decorator

In [31]:
import time


def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f}s")
        return result
    return wrapper


@timer
def slow_square(n):
    total = 0
    for i in range(n):
        total += i * i
    return total


print(slow_square(100_000))

slow_square took 0.005286s
333328333350000


### 2. Temperature class with @property

In [32]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("temperature cannot be below absolute zero")
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9 / 5 + 32


t = Temperature(25)
print(t.celsius, t.fahrenheit)

try:
    t.celsius = -300
except ValueError as e:
    print("Error:", e)

25 77.0
Error: temperature cannot be below absolute zero


### Hard

### 1. @retry decorator factory

In [33]:
def retry(times=3, delay=1):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_error = e
                    print(f"attempt {attempt} failed: {e}")
                    if attempt < times:
                        time.sleep(delay)
            raise last_error
        return wrapper
    return decorator


attempts = {"count": 0}


@retry(times=3, delay=0)
def flaky():
    attempts["count"] += 1
    if attempts["count"] < 3:
        raise ValueError("not ready yet")
    return "success"


print(flaky())

attempt 1 failed: not ready yet
attempt 2 failed: not ready yet
success


### 2. @log_calls stacked with @timer

In [34]:
def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"calling {func.__name__}(args={args}, kwargs={kwargs})")
        return func(*args, **kwargs)
    return wrapper


@log_calls
@timer
def add(a, b):
    return a + b


@timer
@log_calls
def multiply(a, b):
    return a * b


print("--- log_calls then timer ---")
print(add(2, 3))
print("--- timer then log_calls ---")
print(multiply(2, 3))
# add = log_calls(timer(add)): log_calls' print runs first (outermost),
# then timer's print, then the real function.
# multiply = timer(log_calls(multiply)): timer's clock starts first
# (outermost), so it also ends up timing log_calls' own print statement,
# not just the bare function call. Order changes what each wrapper
# actually measures/sees.

--- log_calls then timer ---
calling add(args=(2, 3), kwargs={})
add took 0.000000s
5
--- timer then log_calls ---
calling multiply(args=(2, 3), kwargs={})
multiply took 0.000005s
6


## Challenge Project: Mini Access-Control System + Validated `BankAccount`

### Access control + BankAccount

In [35]:
def requires_role(role):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(user, *args, **kwargs):
            if user.get("role") != role:
                print("Permission denied")
                return None
            return func(user, *args, **kwargs)
        return wrapper
    return decorator


class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    @balance.setter
    def balance(self, value):
        if value < 0:
            raise ValueError("balance cannot be negative")
        self._balance = value

    def deposit(self, amount):
        self.balance = self.balance + amount

    def withdraw(self, amount):
        if amount > self.balance:
            raise ValueError("insufficient funds")
        self.balance = self.balance - amount

    @property
    def is_overdrawn(self):
        return self.balance < 0

    def __repr__(self):
        return f"BankAccount({self.owner!r}, balance={self.balance})"


@requires_role("admin")
def transfer(user, src, dst, amount):
    src.withdraw(amount)
    dst.deposit(amount)
    print(f"Transferred {amount} from {src.owner} to {dst.owner}")


alice_account = BankAccount("alice", 1000)
bob_account = BankAccount("bob", 200)

admin_user = {"name": "root", "role": "admin"}
regular_user = {"name": "ram", "role": "user"}

transfer(regular_user, alice_account, bob_account, 300)
print(alice_account, bob_account)

transfer(admin_user, alice_account, bob_account, 300)
print(alice_account, bob_account)

Permission denied
BankAccount('alice', balance=1000) BankAccount('bob', balance=200)
Transferred 300 from alice to bob
BankAccount('alice', balance=700) BankAccount('bob', balance=500)


### Stretch: audit log + route registry

In [36]:
def audit_log(func):
    @functools.wraps(func)
    def wrapper(user, *args, **kwargs):
        print(f"[audit] {user['name']} invoked {func.__name__}{args}")
        return func(user, *args, **kwargs)
    return wrapper


@audit_log
@requires_role("admin")
def transfer_audited(user, src, dst, amount):
    src.withdraw(amount)
    dst.deposit(amount)
    print(f"Transferred {amount} from {src.owner} to {dst.owner}")


transfer_audited(admin_user, bob_account, alice_account, 50)

routes = {}


def route(path):
    def decorator(func):
        routes[path] = func
        return func
    return decorator


@route("/health")
def health_check():
    return "ok"


print(routes)
print(routes["/health"]())

[audit] root invoked transfer_audited(BankAccount('bob', balance=500), BankAccount('alice', balance=700), 50)
Transferred 50 from bob to alice
{'/health': <function health_check at 0x11151a430>}
ok


# Chapter 5: Object-Oriented Programming (OOP)

## Practice Exercises

### Easy

### 1. Book class with describe()

In [37]:
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author

    def describe(self):
        print(f"{self.title} by {self.author}")


book1 = Book("Dune", "Frank Herbert")
book2 = Book("1984", "George Orwell")
book1.describe()
book2.describe()

Dune by Frank Herbert
1984 by George Orwell


### 2. Class attribute counting instances

In [38]:
class Book:
    count = 0

    def __init__(self, title, author):
        self.title = title
        self.author = author
        Book.count += 1

    def describe(self):
        print(f"{self.title} by {self.author}")


b1 = Book("Dune", "Frank Herbert")
b2 = Book("1984", "George Orwell")
b3 = Book("Foundation", "Isaac Asimov")
print("Total books created:", Book.count)

Total books created: 3


### Medium

### 1. Shape polymorphism

In [39]:
class Shape:
    def area(self):
        raise NotImplementedError


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height


shapes = [Circle(3), Rectangle(4, 5), Circle(1)]
for shape in shapes:
    print(f"{type(shape).__name__} area: {shape.area():.2f}")

Circle area: 28.27
Rectangle area: 20.00
Circle area: 3.14


### 2. BankAccount with private balance

In [40]:
class SimpleBankAccount:
    def __init__(self, balance=0):
        self.__balance = balance

    def deposit(self, amount):
        if amount <= 0:
            print("Invalid deposit amount")
            return
        self.__balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            print("Invalid withdrawal amount")
            return
        if amount > self.__balance:
            print("Insufficient funds")
            return
        self.__balance -= amount

    def get_balance(self):
        return self.__balance


account = SimpleBankAccount(100)
account.deposit(50)
account.withdraw(30)
account.withdraw(1000)
print(account.get_balance())

Insufficient funds
120


### 3. classmethod alternative constructor

In [41]:
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author

    @classmethod
    def from_string(cls, data):
        title, author = data.split("|")
        return cls(title.strip(), author.strip())

    def describe(self):
        print(f"{self.title} by {self.author}")


book = Book.from_string("Dune | Frank Herbert")
book.describe()

Dune by Frank Herbert


### Hard

### 1. Multilevel inheritance Vehicle -> Car -> SportsCar

In [42]:
class Vehicle:
    def __init__(self, make):
        self.make = make

    def describe(self):
        return f"make={self.make}"


class Car(Vehicle):
    def __init__(self, make, doors):
        super().__init__(make)
        self.doors = doors

    def describe(self):
        return super().describe() + f", doors={self.doors}"


class SportsCar(Car):
    def __init__(self, make, doors, top_speed):
        super().__init__(make, doors)
        self.top_speed = top_speed

    def describe(self):
        return super().describe() + f", top_speed={self.top_speed}"


my_car = SportsCar("Ferrari", 2, 340)
print(my_car.make, my_car.doors, my_car.top_speed)
print(my_car.describe())

Ferrari 2 340
make=Ferrari, doors=2, top_speed=340


### 2. Abstract PaymentMethod

In [43]:
from abc import ABC, abstractmethod


class PaymentMethod(ABC):
    @abstractmethod
    def pay(self, amount):
        pass


class CreditCard(PaymentMethod):
    def pay(self, amount):
        print(f"Paid {amount} using Credit Card")


class Esewa(PaymentMethod):
    def pay(self, amount):
        print(f"Paid {amount} using Esewa")


class MobileBanking(PaymentMethod):
    def pay(self, amount):
        print(f"Paid {amount} using Mobile Banking")


payments = [CreditCard(), Esewa(), MobileBanking()]
for method in payments:
    method.pay(500)

try:
    PaymentMethod()
except TypeError as e:
    print("Error:", e)

Paid 500 using Credit Card
Paid 500 using Esewa
Paid 500 using Mobile Banking
Error: Can't instantiate abstract class PaymentMethod with abstract method pay


## Challenge Project: Mini Library Management System

### Library Management System

In [44]:
from abc import ABC, abstractmethod


class LibraryItem(ABC):
    def __init__(self, title):
        self.title = title
        self._checked_out = False

    @abstractmethod
    def describe(self):
        pass


class Book(LibraryItem):
    def __init__(self, title, author):
        super().__init__(title)
        self.author = author

    def describe(self):
        return f"Book: '{self.title}' by {self.author}"


class DVD(LibraryItem):
    def __init__(self, title, runtime_minutes):
        super().__init__(title)
        self.runtime_minutes = runtime_minutes

    def describe(self):
        return f"DVD: '{self.title}' ({self.runtime_minutes} min)"


class Magazine(LibraryItem):
    def __init__(self, title, issue_number):
        super().__init__(title)
        self.issue_number = issue_number

    def describe(self):
        return f"Magazine: '{self.title}' issue #{self.issue_number}"


class Member:
    def __init__(self, name):
        self.name = name
        self.__borrowed_items = []

    def borrow(self, item):
        if item._checked_out:
            raise ValueError(f"'{item.title}' is already checked out")
        item._checked_out = True
        self.__borrowed_items.append(item)

    def return_item(self, item):
        if item not in self.__borrowed_items:
            raise ValueError(f"{self.name} has not borrowed '{item.title}'")
        item._checked_out = False
        self.__borrowed_items.remove(item)

    @property
    def borrowed_items(self):
        return tuple(self.__borrowed_items)


class Library:
    def __init__(self):
        self.items = []
        self.members = []

    def add_item(self, item):
        self.items.append(item)

    def add_member(self, member):
        self.members.append(member)

    def checkout(self, member, item):
        member.borrow(item)

    @classmethod
    def from_catalog(cls, catalog):
        library = cls()
        kind_map = {"book": Book, "dvd": DVD, "magazine": Magazine}
        for entry in catalog:
            entry = dict(entry)
            kind = entry.pop("kind")
            library.add_item(kind_map[kind](**entry))
        return library


catalog = [
    {"kind": "book", "title": "Dune", "author": "Frank Herbert"},
    {"kind": "dvd", "title": "Inception", "runtime_minutes": 148},
    {"kind": "magazine", "title": "National Geographic", "issue_number": 231},
]
library = Library.from_catalog(catalog)
for item in library.items:
    print(item.describe())

ram = Member("Ram")
library.add_member(ram)

dune = library.items[0]
library.checkout(ram, dune)
print(f"{ram.name} borrowed:", [i.title for i in ram.borrowed_items])

try:
    library.checkout(Member("Shyam"), dune)
except ValueError as e:
    print("Error:", e)

ram.return_item(dune)
print(f"{ram.name} borrowed after return:", [i.title for i in ram.borrowed_items])

Book: 'Dune' by Frank Herbert
DVD: 'Inception' (148 min)
Magazine: 'National Geographic' issue #231
Ram borrowed: ['Dune']
Error: 'Dune' is already checked out
Ram borrowed after return: []
